# Comparar base30, mejor_actual y candidato_150 en Colab

Mide los tres modelos con el **harness oficial** en Colab (CPU, 1 hilo) sobre **test_demo** (400 muestras) y estima el score.

1. Carga `colab_comparar.zip` (subida manual o desde Google Drive) y lo descomprime en `/content/modelos/`.
2. Clona los materiales públicos `GDG-UAM/ml-contest-materials` y descarga `test_demo.npz`.
3. Ejecuta `harness.py evaluate --source file:data/test_demo.npz --runs 1` **3 rondas por modelo**, intercalando el orden en cada ronda.
4. Da la **mediana de latencia por muestra** (mediana entre rondas de la mediana de cada ronda), el acierto y el score400.
5. Score estimado: $\text{Score} \approx \sqrt{(60/t)/8}\cdot \text{gm}_f^{\text{val-corr}}$, con la gm_f de la validación corrompida medida en local.

**Uso:** *Entorno de ejecución → Cambiar tipo → CPU* y *Ejecutar todo*. Tarda unos 5–10 min (la primera ronda incluye la compilación de numba).

## 1. Configuración

In [ ]:
USAR_DRIVE = False                                       # True: leer el zip desde Drive en lugar de subirlo
ZIP_DRIVE = "/content/drive/MyDrive/colab_comparar.zip"  # ruta del zip en Drive (si USAR_DRIVE)
RONDAS = 3                                               # rondas por modelo, con el orden intercalado

# gm_f en la validación corrompida (14 250 muestras × 4 versiones), medida en local
GM_F_VAL_CORR = {"base30": 0.520, "mejor_actual": 0.511, "candidato_150": 0.499}
MODELOS = list(GM_F_VAL_CORR)

import glob, json, os, platform, shutil, subprocess, sys, zipfile
MATERIALES = "/content/ml-contest-materials"
MODELOS_DIR = "/content/modelos"
RESULTADOS = "/content/resultados_comparar"

## 2. Zip con los modelos

In [ ]:
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ruta_zip = ZIP_DRIVE
else:
    from google.colab import files
    subido = files.upload()          # elige colab_comparar.zip
    ruta_zip = "/content/" + next(iter(subido))

shutil.rmtree(MODELOS_DIR, ignore_errors=True)
with zipfile.ZipFile(ruta_zip) as z:
    z.extractall(MODELOS_DIR)
for m in MODELOS:
    assert os.path.exists(f"{MODELOS_DIR}/{m}/model.py"), f"falta {m}/model.py en el zip"
    print(m, sorted(os.listdir(f"{MODELOS_DIR}/{m}")))

## 3. Materiales oficiales, datos y dependencias

In [ ]:
if not os.path.isdir(MATERIALES):
    subprocess.run(["git", "clone", "-q", "https://github.com/GDG-UAM/ml-contest-materials.git", MATERIALES], check=True)
ruta = f"{MATERIALES}/data/test_demo.npz"
if not os.path.exists(ruta) or os.path.getsize(ruta) < 1000:
    subprocess.run(["curl", "-sL", "--create-dirs", "-o", ruta,
                    "https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/test_demo.npz"], check=True)
print(f"test_demo.npz: {os.path.getsize(ruta) / 1e6:.2f} MB")

for m in MODELOS:
    req = f"{MODELOS_DIR}/{m}/requirements.txt"
    if os.path.exists(req):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)

import numpy, numba
print(f"Python {platform.python_version()} | numpy {numpy.__version__} | numba {numba.__version__}")
print(subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)'", shell=True, capture_output=True, text=True).stdout)

## 4. Harness (3 rondas por modelo, orden intercalado)

Cada ejecución es una llamada a `harness.py evaluate` con un solo modelo, 1 hilo y `test_demo`. El orden se rota en cada ronda y se invierte en las impares, para que ningún modelo vaya siempre primero o último.

In [ ]:
env = {**os.environ}
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    env[v] = "1"

def harness(m, ronda):
    salida = f"{RESULTADOS}/r{ronda}_{m}"
    shutil.rmtree(salida, ignore_errors=True)
    cmd = [sys.executable, "harness.py", "evaluate", "--model", f"{MODELOS_DIR}/{m}/model.py",
           "--source", "file:data/test_demo.npz", "--runs", "1", "--out", salida]
    r = subprocess.run(cmd, cwd=MATERIALES, env=env, capture_output=True, text=True)
    runs = sorted(glob.glob(f"{salida}/*/run_0.json"))
    assert r.returncode == 0 and runs, f"{m}: el harness falló\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}"
    return json.load(open(runs[0]))

RES = []
for ronda in range(RONDAS):
    k = ronda % len(MODELOS)
    orden = MODELOS[k:] + MODELOS[:k]
    if ronda % 2:
        orden = orden[::-1]
    for m in orden:
        j = harness(m, ronda)
        lat = j.get("latency_s") or {}
        RES.append({"ronda": ronda, "modelo": m, "estado": j["status"], "aciertos": j["n_correct"],
                    "acierto": j["accuracy_all"], "score400": j["score"], "setup (s)": j["setup_time_s"],
                    "lat. mediana (µs)": lat.get("median", float("nan")) * 1e6,
                    "lat. media (µs)": lat.get("mean", float("nan")) * 1e6,
                    "incidencias": j["n_exception"] + j["n_invalid"] + j["n_soft_timeout"] + j["n_hard_timeout"]})
        print(f"ronda {ronda} | {m:14s} | {RES[-1]['lat. mediana (µs)']:7.1f} µs | {j['n_correct']}/400 | score400 {j['score']:.3f}")

## 5. Resultados y score estimado

In [ ]:
import numpy as np
import pandas as pd

R = pd.DataFrame(RES)
display(R.round(3))

ref = R[R.modelo == "base30"].set_index("ronda")["lat. mediana (µs)"]
filas = []
for m in MODELOS:
    d = R[R.modelo == m].set_index("ronda")
    t = float(d["lat. mediana (µs)"].median()) * 1e-6        # mediana entre rondas de la mediana por muestra
    N = min(1e7, 60.0 / t)
    g = GM_F_VAL_CORR[m]
    filas.append({"modelo": m, "lat. mediana (µs)": t * 1e6,
                  "×base30 (emparejada)": float((d["lat. mediana (µs)"] / ref).median()),
                  "aciertos": int(d["aciertos"].iloc[0]), "acierto": float(d["acierto"].iloc[0]),
                  "score400": float(d["score400"].iloc[0]), "gm_f val-corr": g,
                  "N en 60 s": N, "score estimado": np.sqrt(N / 8) * g,
                  "estable": d["aciertos"].nunique() == 1 and (d["estado"] == "ok").all() and d["incidencias"].sum() == 0})
T = pd.DataFrame(filas).set_index("modelo")
display(T.round({"lat. mediana (µs)": 1, "×base30 (emparejada)": 3, "acierto": 4, "score400": 3, "N en 60 s": 0,
                 "score estimado": 1}))

mejor = T["score estimado"].idxmax()
print(f"Mejor score estimado: {mejor} ({T.loc[mejor, 'score estimado']:.1f})")
print(f"candidato_150 / mejor_actual: latencia ×{T.loc['candidato_150', 'lat. mediana (µs)'] / T.loc['mejor_actual', 'lat. mediana (µs)']:.3f}, "
      f"score estimado ×{T.loc['candidato_150', 'score estimado'] / T.loc['mejor_actual', 'score estimado']:.3f}")
print("Para que candidato_150 iguale a mejor_actual necesita latencia ≤ "
      f"{T.loc['mejor_actual', 'lat. mediana (µs)'] * (0.499 / 0.511) ** 2:.1f} µs  (×{(0.499 / 0.511) ** 2:.3f}).")
print("Referencia local: aciertos test_demo base30 330, mejor_actual 330, candidato_150 327 (iguales en todas las rondas).")